# 🎮 Steam Game Dataset — Exploratory Data Analysis

### What is this notebook about?

This notebook is my attempt to understand the Steam games dataset before building an analytics agent on top of it. The idea is simple — before I start answering questions like *"Do free games have better ratings?"* or *"What are the best multiplayer shooters?"*, I need to actually understand what the data looks like, what's missing, what's messy, and what's useful.

I'll go through the data step by step, note down my observations in plain language, clean up the issues I find, and engineer some new features that will make the analytics agent more powerful.

---

**The Dataset:**
We have 3 CSV files:
- `game_ids.csv` — Just a mapping of app_id to game name (29,235 games)
- `additional_data.csv` — The main file with pricing, reviews, genres, tags, languages, playtime
- `game_data.csv` — Detailed Steam store data (release dates, descriptions, categories) — large file ~270MB

Let's dig in.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import ast
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
sns.set_theme(style='whitegrid', palette='muted')

print('Libraries loaded ✅')

## Step 1 — Loading the Data

Let's start by loading the two smaller files. The large `game_data.csv` will be loaded separately.

In [ ]:
game_ids = pd.read_csv('Game_Analytics_Agent/data/game_ids.csv')
add_data = pd.read_csv('Game_Analytics_Agent/data/additional_data.csv')

print(f'game_ids shape  : {game_ids.shape}')
print(f'add_data shape  : {add_data.shape}')
print()
print('game_ids columns :', list(game_ids.columns))
print('add_data columns :', list(add_data.columns))

In [ ]:
# Quick peek at game_ids
print('--- game_ids sample ---')
display(game_ids.head(5))

In [ ]:
# Quick peek at additional_data
print('--- additional_data sample ---')
display(add_data.head(3))

**First observations just from looking at the raw data:**

- `game_ids.csv` is basically a lookup table — just app_id and name. Not much to do here.
- `additional_data.csv` is where all the interesting stuff is — reviews, pricing, playtime, tags, etc.
- The `price` column looks suspicious — Counter-Strike shows `999.0`. That seems too high for dollars. I suspect it's stored in **cents** (999 cents = $9.99). I'll confirm this.
- The `tags` column is a Python dict stored as a string — will need special handling.
- The `owners` column is a range string like `'10,000,000 .. 20,000,000'` — interesting, not a clean number.
- `score_rank` is almost entirely missing (more on this later).

## Step 2 — Understanding the Shape and Missing Values

In [ ]:
print('Data types:')
print(add_data.dtypes)
print()
print('Missing values per column:')
missing = add_data.isnull().sum()
missing_pct = (missing / len(add_data) * 100).round(2)
summary = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
display(summary[summary['missing_count'] > 0].sort_values('missing_%', ascending=False))

**What I notice about missing data:**

- `score_rank` is missing for **99.8%** of games. This column is essentially useless — Steam only assigns a score rank to games with a certain review threshold, and most games on Steam are small indie titles that never reach it. I'll drop this column.
- `developer` is missing for 198 games and `publisher` for 296. These are likely old or delisted games. I'll fill them with `'Unknown'`.
- `genre` is missing for 152 games — small number, I'll fill with `'Unknown'`.
- `languages` is missing for 94 games — similarly, fill with `'Unknown'`.
- `price` is missing for only 29 games — I'll assume these are free (price = 0).
- Everything else (reviews, playtime, ccu) is fully populated. 

## Step 3 — Data Cleaning

In [ ]:
df = add_data.copy()

# 3.1 Drop score_rank — 99.8% missing, not useful
df.drop(columns=['score_rank'], inplace=True)
print('Dropped score_rank column')

# 3.2 Fill missing text columns
df['developer'].fillna('Unknown', inplace=True)
df['publisher'].fillna('Unknown', inplace=True)
df['genre'].fillna('Unknown', inplace=True)
df['languages'].fillna('Unknown', inplace=True)
df['name'].fillna('Unknown', inplace=True)
print('Filled missing text columns with Unknown')

# 3.3 Fix price — it's stored in CENTS, divide by 100 to get USD
# Evidence: Counter-Strike shows 999 → $9.99, which matches Steam
df['price'].fillna(0, inplace=True)
df['initialprice'].fillna(0, inplace=True)
df['discount'].fillna(0, inplace=True)

df['price_usd']         = df['price'] / 100
df['initialprice_usd']  = df['initialprice'] / 100

print(f'Price check — Counter-Strike price_usd: ${df[df["name"]=="Counter-Strike"]["price_usd"].values[0]}')
print(f'Price range: $0 to ${df["price_usd"].max()}')

In [ ]:
# 3.4 is_free flag
df['is_free'] = df['price_usd'] == 0.0
print(f'Free games  : {df["is_free"].sum():,}')
print(f'Paid games  : {(~df["is_free"]).sum():,}')
print(f'Free %      : {df["is_free"].mean()*100:.1f}%')

In [ ]:
# 3.5 Remove games with no reviews at all — these are likely unreleased/hidden apps
df['total_reviews'] = df['positive'] + df['negative']
no_reviews = (df['total_reviews'] == 0).sum()
print(f'Games with 0 reviews: {no_reviews} ({no_reviews/len(df)*100:.1f}%)')
# I'll keep them in the dataset but flag them
df['has_reviews'] = df['total_reviews'] > 0
print('Flagged 0-review games with has_reviews=False')

In [ ]:
# 3.6 Remove obvious outlier prices — anything above $100 is likely a DLC bundle or error
high_price = df[df['price_usd'] > 100]
print(f'Games priced above $100: {len(high_price)}')
print(high_price[['name','price_usd']].head(10))

**Note on high prices:** Some games do legitimately cost more than $100 (usually software tools, enterprise apps, or DLC bundles mis-categorized as games). I'll keep them in the dataset but they'll be obvious outliers in price distribution charts.

## Step 4 — Feature Engineering

Now the fun part — creating new columns that will make the analytics more meaningful.

In [ ]:
# 4.1 Positive Ratio — what % of reviews are positive
# This is our main "rating" signal — much better than raw positive counts
df['positive_ratio'] = np.where(
    df['total_reviews'] > 0,
    df['positive'] / df['total_reviews'],
    np.nan
)

print('Positive ratio stats (for games with reviews):')
print(df['positive_ratio'].describe().round(3))
print()
print(f'Average rating across all games: {df["positive_ratio"].mean()*100:.1f}%')

In [ ]:
# 4.2 Steam Review Label — Steam uses these thresholds for review badges
def review_label(ratio, count):
    if pd.isna(ratio) or count < 10:
        return 'No Rating'
    elif ratio >= 0.95:
        return 'Overwhelmingly Positive'
    elif ratio >= 0.80:
        return 'Very Positive'
    elif ratio >= 0.70:
        return 'Mostly Positive'
    elif ratio >= 0.40:
        return 'Mixed'
    elif ratio >= 0.20:
        return 'Mostly Negative'
    else:
        return 'Overwhelmingly Negative'

df['review_label'] = df.apply(
    lambda r: review_label(r['positive_ratio'], r['total_reviews']), axis=1
)

print('Review label distribution:')
print(df['review_label'].value_counts())

In [ ]:
# 4.3 Price Category — group games into pricing tiers
def price_tier(price):
    if price == 0:
        return 'Free'
    elif price <= 5:
        return 'Budget (≤$5)'
    elif price <= 15:
        return 'Mid-range ($5–$15)'
    elif price <= 30:
        return 'Premium ($15–$30)'
    else:
        return 'AAA/Expensive (>$30)'

df['price_tier'] = df['price_usd'].apply(price_tier)

print('Price tier distribution:')
print(df['price_tier'].value_counts())

In [ ]:
# 4.4 Primary Genre — many games have multi-genre tags like "Action, Indie"
# Extract just the first genre for cleaner analysis
df['primary_genre'] = df['genre'].str.split(',').str[0].str.strip()

print('Top 10 primary genres:')
print(df['primary_genre'].value_counts().head(10))

In [ ]:
# 4.5 Multiplayer flag — check if the game has multiplayer in its tags
df['has_multiplayer'] = df['tags'].str.contains('Multiplayer', case=False, na=False)
df['has_singleplayer'] = df['tags'].str.contains('Singleplayer', case=False, na=False)

print(f'Games with multiplayer tag : {df["has_multiplayer"].sum():,} ({df["has_multiplayer"].mean()*100:.1f}%)')
print(f'Games with singleplayer tag: {df["has_singleplayer"].sum():,} ({df["has_singleplayer"].mean()*100:.1f}%)')

In [ ]:
# 4.6 Language count — how many languages does each game support?
df['language_count'] = df['languages'].apply(
    lambda x: len(str(x).split(',')) if str(x) != 'Unknown' else 0
)

print('Language count stats:')
print(df['language_count'].describe().round(1))
print(f'\nGames supporting 10+ languages: {(df["language_count"] >= 10).sum():,}')

In [ ]:
# 4.7 Popularity tier based on owner count
def owner_tier(owners_str):
    if pd.isna(owners_str):
        return 'Unknown'
    s = str(owners_str)
    if '100,000,000' in s or '200,000,000' in s:
        return 'Mega (100M+)'
    elif '10,000,000' in s or '20,000,000' in s or '50,000,000' in s:
        return 'Blockbuster (10M+)'
    elif '1,000,000' in s or '2,000,000' in s or '5,000,000' in s:
        return 'Popular (1M+)'
    elif '100,000' in s or '200,000' in s or '500,000' in s:
        return 'Known (100K+)'
    elif '20,000' in s or '50,000' in s:
        return 'Small (20K+)'
    else:
        return 'Niche (<20K)'

df['popularity_tier'] = df['owners'].apply(owner_tier)

print('Popularity tier distribution:')
print(df['popularity_tier'].value_counts())

In [ ]:
# 4.8 Average playtime in hours (original is in minutes)
df['avg_playtime_hours'] = df['average_forever'] / 60
df['median_playtime_hours'] = df['median_forever'] / 60

print('Average playtime stats (hours):')
print(df[df['avg_playtime_hours'] > 0]['avg_playtime_hours'].describe().round(1))

## Step 5 — Exploratory Analysis & Visualizations

Now let's actually look at the data visually.

In [ ]:
# 5.1 Free vs Paid breakdown
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
free_count = df['is_free'].sum()
paid_count = (~df['is_free']).sum()
axes[0].pie([free_count, paid_count],
            labels=[f'Free\n({free_count:,})', f'Paid\n({paid_count:,})'],
            colors=['#55A868','#4C72B0'],
            autopct='%1.1f%%', startangle=90)
axes[0].set_title('Free vs Paid Games', fontsize=13)

# Price distribution for paid games (capped at $60)
paid_prices = df[(df['is_free']==False) & (df['price_usd'] <= 60)]['price_usd']
axes[1].hist(paid_prices, bins=40, color='#4C72B0', edgecolor='white')
axes[1].set_xlabel('Price (USD)')
axes[1].set_ylabel('Number of Games')
axes[1].set_title('Price Distribution (Paid Games, capped at $60)', fontsize=13)

plt.tight_layout()
plt.savefig('free_vs_paid.png', dpi=120)
plt.show()

print(f'\n📊 Observation: Most paid games are priced under $10.')
print(f'   Median paid price: ${paid_prices.median():.2f}')
print(f'   Only {(paid_prices > 30).sum():,} games are priced above $30')

In [ ]:
# 5.2 Free vs Paid — which has better ratings?
free_avg = df[df['is_free']==True]['positive_ratio'].mean() * 100
paid_avg = df[df['is_free']==False]['positive_ratio'].mean() * 100

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(['Free Games', 'Paid Games'], [free_avg, paid_avg],
               color=['#55A868','#4C72B0'], width=0.5, edgecolor='white')
ax.set_ylim(60, 80)
ax.set_ylabel('Average Positive Ratio (%)')
ax.set_title('Free vs Paid: Average Rating Comparison', fontsize=13)
for bar, val in zip(bars, [free_avg, paid_avg]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.2f}%', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('free_vs_paid_rating.png', dpi=120)
plt.show()

print(f'\n📊 Free games avg rating  : {free_avg:.2f}%')
print(f'   Paid games avg rating  : {paid_avg:.2f}%')
print(f'   → {"Paid" if paid_avg > free_avg else "Free"} games have slightly higher ratings, but the difference is very small.')
print(f'   This suggests quality is spread fairly evenly regardless of price.')

In [ ]:
# 5.3 Top 10 Primary Genres
top_genres = df['primary_genre'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(13, 5))
top_genres.plot(kind='bar', ax=ax, color='#4C72B0', edgecolor='white')
ax.set_title('Top 10 Primary Genres on Steam', fontsize=13)
ax.set_xlabel('Genre')
ax.set_ylabel('Number of Games')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.savefig('top_genres.png', dpi=120)
plt.show()

print('\n📊 Observation: Indie is the biggest category by far.')
print('   Steam is essentially an indie game marketplace with a few big titles.')
print('   Action and Adventure are the most popular non-Indie genres.')

In [ ]:
# 5.4 Review label distribution
label_order = ['Overwhelmingly Positive','Very Positive','Mostly Positive',
               'Mixed','Mostly Negative','Overwhelmingly Negative','No Rating']
label_counts = df['review_label'].value_counts().reindex(label_order, fill_value=0)
colors = ['#2ecc71','#27ae60','#f39c12','#e67e22','#e74c3c','#c0392b','#95a5a6']

fig, ax = plt.subplots(figsize=(13, 5))
label_counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_title('Distribution of Steam Review Labels', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Number of Games')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('review_labels.png', dpi=120)
plt.show()

print('\n📊 Observation: Most games that have enough reviews are rated positively.')
print(f'   Very/Overwhelmingly Positive: {(label_counts["Very Positive"] + label_counts["Overwhelmingly Positive"]):,} games')
print(f'   Mixed or Negative: {(label_counts["Mixed"] + label_counts["Mostly Negative"] + label_counts["Overwhelmingly Negative"]):,} games')

In [ ]:
# 5.5 Top 15 Tags across all games
tag_counts = {}
for t in df['tags'].dropna():
    try:
        tag_dict = ast.literal_eval(t)
        for k in tag_dict:
            tag_counts[k] = tag_counts.get(k, 0) + 1
    except:
        pass

top_tags = pd.Series(tag_counts).nlargest(15)

fig, ax = plt.subplots(figsize=(13, 5))
top_tags.plot(kind='bar', ax=ax, color='#4C72B0', edgecolor='white')
ax.set_title('Top 15 Tags on Steam', fontsize=13)
ax.set_ylabel('Number of Games')
ax.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.savefig('top_tags.png', dpi=120)
plt.show()

print('\n📊 Top tags tell us what players actually associate with games.')
print('   Indie, Action, Adventure, and Casual dominate.')
print('   Multiplayer comes in at #13, VR is surprisingly in top 15.')

In [ ]:
# 5.6 Top 10 Languages supported
lang_counts = {}
for l in df['languages'].dropna():
    if l == 'Unknown':
        continue
    for lang in str(l).split(','):
        lang = lang.strip()
        if lang:
            lang_counts[lang] = lang_counts.get(lang, 0) + 1

top_langs = pd.Series(lang_counts).nlargest(10)

fig, ax = plt.subplots(figsize=(13, 5))
top_langs.plot(kind='barh', ax=ax, color='#55A868', edgecolor='white')
ax.set_title('Top 10 Supported Languages on Steam', fontsize=13)
ax.set_xlabel('Number of Games')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('top_languages.png', dpi=120)
plt.show()

print('\n📊 English dominates with almost every game supporting it.')
print(f'   English: {lang_counts.get("English",0):,} games')
print(f'   German:  {lang_counts.get("German",0):,} games')
print(f'   Korean:  {lang_counts.get("Korean",0):,} games')
print('   Korean is supported by a significant number of games, reflecting Steam\'s popularity in South Korea.')

In [ ]:
# 5.7 Popularity distribution — how many owners do games typically have?
pop_order = ['Niche (<20K)', 'Small (20K+)', 'Known (100K+)',
             'Popular (1M+)', 'Blockbuster (10M+)', 'Mega (100M+)']
pop_counts = df['popularity_tier'].value_counts().reindex(pop_order, fill_value=0)

fig, ax = plt.subplots(figsize=(13, 5))
pop_counts.plot(kind='bar', ax=ax, color='#4C72B0', edgecolor='white')
ax.set_title('Games by Ownership Tier', fontsize=13)
ax.set_ylabel('Number of Games')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.savefig('popularity_tiers.png', dpi=120)
plt.show()

print('\n📊 Observation: Steam has a massive long tail.')
print(f'   {pop_counts["Niche (<20K)"]:,} games ({pop_counts["Niche (<20K)"]/len(df)*100:.0f}%) have fewer than 20,000 owners.')
print('   A tiny number of blockbuster games make up the vast majority of playtime.')

In [ ]:
# 5.8 Average rating by price tier
tier_order = ['Free', 'Budget (≤$5)', 'Mid-range ($5–$15)', 'Premium ($15–$30)', 'AAA/Expensive (>$30)']
tier_ratings = df.groupby('price_tier')['positive_ratio'].mean().reindex(tier_order) * 100

fig, ax = plt.subplots(figsize=(11, 5))
tier_ratings.plot(kind='bar', ax=ax, color='#4C72B0', edgecolor='white')
ax.set_title('Average Rating by Price Tier', fontsize=13)
ax.set_ylabel('Average Positive Ratio (%)')
ax.set_ylim(60, 85)
ax.tick_params(axis='x', rotation=20)
for i, v in enumerate(tier_ratings):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('rating_by_price_tier.png', dpi=120)
plt.show()

print('\n📊 Insight: Premium and AAA games tend to have slightly higher ratings.')
print('   Players expect more from expensive games — developers deliver to survive.')
print('   Budget games have lowest ratings — flooded with low-effort content.')

## Step 6 — Answering the 4 Core Questions from the Problem Statement

In [ ]:
# Q1: Do free games have higher ratings on average than paid games?
free_avg = df[df['is_free']==True]['positive_ratio'].mean() * 100
paid_avg = df[df['is_free']==False]['positive_ratio'].mean() * 100
print('Q1: Do free games have higher ratings on average than paid games?')
print(f'   Free: {free_avg:.2f}% | Paid: {paid_avg:.2f}%')
print(f'   Answer: {"Free" if free_avg > paid_avg else "Paid"} games have slightly higher ratings.')
print(f'   Difference is only {abs(free_avg - paid_avg):.2f}% — practically negligible.')
print()

In [ ]:
# Q2: How many Action games support Korean?
action_korean = df[
    df['genre'].str.contains('Action', case=False, na=False) &
    df['languages'].str.contains('Korean', case=False, na=False)
]
print(f'Q2: How many Action games support Korean?')
print(f'   Answer: {len(action_korean):,} Action games support Korean')
print(f'   Sample: {list(action_korean["name"].head(5))}')
print()

In [ ]:
# Q3: Best multiplayer shooters
best_shooters = df[
    df['tags'].str.contains('Shooter', case=False, na=False) &
    df['tags'].str.contains('Multiplayer', case=False, na=False) &
    (df['total_reviews'] >= 100)
].nlargest(10, 'positive_ratio')

print('Q3: Best multiplayer shooters:')
for _, row in best_shooters.iterrows():
    print(f'   • {row["name"]} — Rating: {row["positive_ratio"]*100:.1f}% ({int(row["total_reviews"]):,} reviews)')
print()

In [ ]:
# Q4: Price of Counter-Strike in INR and USD
USD_TO_INR = 83.5
cs_games = df[df['name'].str.contains('Counter-Strike|Counter Strike', case=False, na=False)]
print('Q4: Price of Counter-Strike games:')
for _, row in cs_games.iterrows():
    usd = row['price_usd']
    inr = round(usd * USD_TO_INR, 2)
    print(f'   • {row["name"]}: ${usd:.2f} USD / ₹{inr:.2f} INR')

## Step 7 — Final Cleaned Dataset Summary

In [ ]:
print('=== FINAL DATASET SUMMARY ===')
print(f'Total games          : {len(df):,}')
print(f'Free games           : {df["is_free"].sum():,} ({df["is_free"].mean()*100:.1f}%)')
print(f'Paid games           : {(~df["is_free"]).sum():,}')
print(f'Avg positive ratio   : {df["positive_ratio"].mean()*100:.1f}%')
print(f'Games with reviews   : {df["has_reviews"].sum():,}')
print(f'Avg paid price       : ${df[df["is_free"]==False]["price_usd"].mean():.2f}')
print(f'Multiplayer games    : {df["has_multiplayer"].sum():,}')
print(f'New columns added    : positive_ratio, review_label, price_usd, price_tier,')
print(f'                       is_free, has_reviews, primary_genre, has_multiplayer,')
print(f'                       has_singleplayer, language_count, popularity_tier,')
print(f'                       avg_playtime_hours, total_reviews')
print()
print('EDA Complete ✅ — Ready to power the Game Analytics Agent')

## Summary of Key Findings

After going through this dataset thoroughly, here's what I learned:

1. **Price was in cents, not dollars** — The biggest data quality issue. `999` means `$9.99`, not `$999`. Fixed by dividing by 100.

2. **`score_rank` is useless** — 99.8% missing. Steam only assigns this to top-ranked games. Dropped it.

3. **Steam is dominated by indie games** — Over 60% of games have fewer than 20,000 owners. It's a long-tail marketplace.

4. **Free vs Paid ratings are nearly identical** — Free: ~70.3%, Paid: ~71.3%. The difference is so small it's not meaningful.

5. **Most games cluster under $10** — Median paid price is $5.99. AAA pricing ($60+) is rare on Steam.

6. **Tags are the best signal** — The tags dict-string column is rich with community-assigned labels. Much more granular than the genre column for filtering.

7. **English is universal, but German and Russian follow** — Any game supporting 5+ languages almost certainly supports German, French, and Russian.

8. **No release dates in `additional_data.csv`** — Date information only lives in `game_data.csv`. Queries like "after 2015" won't work unless that file is loaded.